# *This notebook provides a workflow to carry out a benchmark on all the omics datasets considered in order. Two hyperparameters are tested*:
- the damping factor *d*
- the Pearson's correlation value *r* that has been applied on the datasets to filter out the data during the preprocessing

A report is then written to mention which omics dataset is the best for the biological process of interest.

### *Importing the required libraries*

In [ ]:
import glob
import networkx as nx
import os
import pandas as pd
import re
import shutil
import sys
sys.path.append("../scripts")

import omics_analysis
from tqdm import tqdm

### *Reading the training genes and setting the input parameters*

In [ ]:
# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

# Setting the process of interest
process = "stalk_cell"

# Listing the omics datasets
datasets = ["E-GEOD-18913", "E-GEOD-37741", "E-GEOD-45750", "E-MTAB-7774", "GSE77597", "GSE89174", "GSE221860", "GSE221861"]

# Setting the path to the omics folder 
path_data = f"../graphs/omics"

# Setting the path to the hyperparameters folder
if not os.path.exists("../graphs/omics/hyperparameters"):
    os.mkdir("../graphs/omics/hyperparameters")

path_hyperparameters = f"../graphs/omics/hyperparameters/{process}"
if not os.path.exists(f"{path_hyperparameters}"):
    os.mkdir(f"{path_hyperparameters}")

# Setting the path to the results folder
    if not os.path.exists("../results"):
        os.mkdir("../results")

### *Initializing the benchmark*

In [ ]:
# Initiating the loop
for dataset in tqdm(datasets, desc = "Benchmarking omics datasets ...", position = 0):

    # Listing all the graph files to consider for the input dataset 
    files = glob.glob(f"{path_data}/{dataset}*.graphml")

    # Create a folder for the upcoming threshold benchmark
    path_benchmark = f"../results/{process}"
    path_benchmark_dataset = f"{path_benchmark}/{dataset}"
    path_results = f"{path_benchmark_dataset}/Benchmark_correlation_and_DF"

    if not os.path.exists(f"{path_benchmark}"):
        os.mkdir(f"{path_benchmark}")

    if not os.path.exists(f"{path_benchmark_dataset}"):
        os.mkdir(f"{path_benchmark_dataset}")

    if not os.path.exists(f"{path_results}"):
        os.mkdir(f"{path_results}")
    else:
        shutil.rmtree(f"{path_results}")
        os.mkdir(f"{path_results}")
   
    # Initializing the workflow    
    for file in tqdm(files, desc = "Processing correlation thresholds files ...", position = 1):

        # Reading the network file and retrieving the correlation threshold that has been applied to generate the graph file
        G = nx.read_graphml(file)
        filename = os.path.basename(file)
        match = re.search(f"{dataset}_(.*)_graph.graphml", filename)

        if match:
            correlation_threshold = match.group(1)
            correlation_threshold = correlation_threshold.replace(".", "")

        # Determining the output file name
        output_file = f"{path_results}/benchmark_{dataset}_corr_{correlation_threshold}.csv"

    # Running a benchmark on the damping factor value to find the best
        benchmark_results = omics_analysis.analyse(G, genes, correlation_threshold, output_file)    

    # Analyzing the benchmark results : we need to determine the best threshold for the scores and the best damping factor
    correlation_thresholds = []
    AUCs = []
    Damping_factors = []

    files = glob.glob(f"{path_results}/*.csv")
    for file in files:
        results = pd.read_csv(file)
        correlation_thresholds.append(int(results.loc[:, "correlation_threshold"].mean()))
    
        best_df = results[results["auc_roc"] == results["auc_roc"].max()]
        for i in best_df.itertuples():
            Damping_factors.append(i[1])
            AUCs.append(i[7])

    benchmark_process = pd.DataFrame({"correlation_threshold": correlation_thresholds,
        "Best_AUC": AUCs,
        "Best_DF": Damping_factors})

    benchmark_process.to_csv(f"{path_results}/Benchmark_damping_factor_{dataset}_{process}.csv",
        sep = ",", index = False)

    # Using the benchmark to find the hyperparameters (correlation threshold and damping factor)
    hyperparameters_df = benchmark_process[benchmark_process["Best_AUC"] == benchmark_process["Best_AUC"].max()]
    hyperparameters_df.to_csv(f"{path_hyperparameters}/Hyperparameters_{dataset}.csv", sep = ",", index = False)

### *Analyzing the benchmark results to find the best omics dataset for the process considered*

In [ ]:
# Analyzing the benchmark: determining the best omics dataset for the process considered
pattern = r"\_(.*?)\_"

# Step 1: Cleaning the files
files = glob.glob(f"../graphs/omics/hyperparameters/{process}/*.csv")
for file in files:

    # Extracting the dataset name
    filename = os.path.basename(file)
    dataset = re.search(pattern, filename).group(1)

    # Adding a column for the name of the dataset
    df = pd.read_csv(file)
    df["Dataset"] = dataset

    # Reordering the columns
    new_cols = ["Dataset", "correlation_threshold", "Best_AUC", "Best_DF"]
    df = df.reindex(columns = new_cols)

    # Writting the actual correlation threshold value
    df["correlation_threshold"] = "0." + df["correlation_threshold"].astype(str)

    # Saving the clean file
    df.to_csv(file, sep = ",", index = False)

# Step 2: Concatenating the files and retrieving the best dataset for the process considered
files = glob.glob(f"../graphs/omics/hyperparameters/{process}/*.csv")

df_best = pd.concat(map(pd.read_csv, files), ignore_index = True)
df_best = df_best[df_best["Best_AUC"] == df_best["Best_AUC"].max()]

# Step 3: Writting a .txt file storing the best omics dataset parameters
with open(f"../graphs/omics/hyperparameters/Best_omics_{process}.txt", 
        "w") as f_out:
    f_out.write(f"Best omics dataset for {process} process: {df_best['Dataset'].values[0]}\n")
    f_out.write(f"---> correlation threshold: {df_best['correlation_threshold'].values[0]}\n")
    f_out.write(f"---> damping factor: {df_best['Best_DF'].values[0]}\n")

print(f"Best omics dataset for {process} process: {df_best['Dataset'].values[0]}")
print(f"---> correlation threshold: {df_best['correlation_threshold'].values[0]}")
print(f"---> damping factor: {df_best['Best_DF'].values[0]}")